Flow chart: https://lucid.app/lucidchart/7c3de4c8-d724-46d7-a295-d4f7085c9dcb/edit?viewport_loc=249%2C-129%2C4531%2C2282%2CVWPGW8r3pMRZ&invitationId=inv_a2f03941-b3b9-4d7e-93e3-211d83de0824

Imports

In [ ]:
 !pip install simpy

In [ ]:
import simpy
import random
from dataclasses import dataclass
from collections import defaultdict, deque

Parameters

In [ ]:
# ---------------------------
# GLOBAL SETTINGS
# ---------------------------

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

SIM_YEARS = 70
DAYS_PER_YEAR = 365
SIM_DAYS = SIM_YEARS * DAYS_PER_YEAR

# ---------------------------
# ARRIVAL SETTINGS
# ---------------------------

DAILY_PATIENTS = 200   # fixed 200 patients/day

# Two patient types for now
PATIENT_TYPE_PROBS = {
    "outpatient": 0.70,   # scheduled arrivals
    "drop_in": 0.30       # random arrivals
}

# Destination/provider for incoming patients
DESTINATION_PROBS = {
    "pcp": 0.35,
    "gynecologist": 0.25,
    "specialist": 0.20,
    "er": 0.20
}

# Among ER arrivals, critical or not
ER_CRITICAL_PROB = 0.30

# Probability a scheduled outpatient actually shows up
OUTPATIENT_SHOW_PROB = 1.00   # keep at 1 for now; later can lower for no-shows

# ---------------------------
# DAILY CAPACITY
# ---------------------------

PCP_DAILY_CAPACITY = 40
GYN_DAILY_CAPACITY = 30
SPECIALIST_DAILY_CAPACITY = 20
ER_DAILY_CAPACITY = 25

Patient class (minimal for now)

In [ ]:
@dataclass
class Patient:
    patient_id: int
    day_created: int

    patient_type: str          # outpatient or drop_in
    destination: str           # pcp / gynecologist / specialist / er

    critical_status: bool = False
    scheduled_day: int = 0

    wait_days: int = 0
    return_count: int = 0

Helper Functions

In [ ]:
def is_weekday(day):
    """
    Treat day 0 as Monday.
    0=Mon, 1=Tue, 2=Wed, 3=Thu, 4=Fri, 5=Sat, 6=Sun
    """
    return (day % 7) < 5


def next_weekday(day):
    """
    Return the next weekday after 'day'.
    If next day is weekend, skip to Monday.
    """
    d = day + 1
    while not is_weekday(d):
        d += 1
    return d


def sample_patient_type():
    types = list(PATIENT_TYPE_PROBS.keys())
    probs = list(PATIENT_TYPE_PROBS.values())
    return random.choices(types, weights=probs, k=1)[0]


def sample_destination():
    dests = list(DESTINATION_PROBS.keys())
    probs = list(DESTINATION_PROBS.values())
    return random.choices(dests, weights=probs, k=1)[0]


def sample_er_critical():
    return random.random() < ER_CRITICAL_PROB

Initialization

In [ ]:
def initialize_state():
    return {
        # Queues for patients waiting to be seen TODAY
        "pcp_today": deque(),
        "gyn_today": deque(),
        "specialist_today": deque(),
        "er_today": deque(),

        # Scheduled outpatients for future dates
        "future_schedule": defaultdict(list),

        # Critical ER patients who return next weekday as drop-ins
        "er_return_next_day": defaultdict(list),

        # Patient registry
        "all_patients": [],

        # Metrics
        "patients_created": 0,

        "created_by_type": defaultdict(int),
        "created_by_destination": defaultdict(int),

        "seen_by_destination": defaultdict(int),
        "not_seen_by_destination": defaultdict(int),

        "converted_dropin_to_outpatient": 0,
        "critical_er_returned": 0,
        "noncritical_er_scheduled": 0,

        "outpatient_showups": 0,
        "outpatient_noshows": 0,

        # Daily logs
        "daily_log": defaultdict(list)
    }

Logging helper

In [ ]:
def log_day(state, day, message):
    state["daily_log"][day].append(message)

Route patients into today's queue

In [ ]:
def add_patient_to_today_queue(patient, state):
    if patient.destination == "pcp":
        state["pcp_today"].append(patient)
    elif patient.destination == "gynecologist":
        state["gyn_today"].append(patient)
    elif patient.destination == "specialist":
        state["specialist_today"].append(patient)
    elif patient.destination == "er":
        state["er_today"].append(patient)

Process scheduled outpatients for today

In [ ]:
def release_scheduled_patients_for_today(day, state):
    """
    Move scheduled outpatients into today's provider queues.
    """
    scheduled_today = state["future_schedule"][day]
    state["future_schedule"][day] = []

    for patient in scheduled_today:
        if random.random() < OUTPATIENT_SHOW_PROB:
            state["outpatient_showups"] += 1
            add_patient_to_today_queue(patient, state)
            log_day(
                state, day,
                f"Patient {patient.patient_id} scheduled outpatient arrives for {patient.destination}"
            )
        else:
            state["outpatient_noshows"] += 1
            log_day(
                state, day,
                f"Patient {patient.patient_id} scheduled outpatient no-show for {patient.destination}"
            )

Process returning critical ER patients

In [ ]:
def release_returning_er_patients(day, state):
    """
    Critical ER patients who could not be seen return next weekday as drop-ins.
    """
    returning_today = state["er_return_next_day"][day]
    state["er_return_next_day"][day] = []

    for patient in returning_today:
        patient.wait_days += 1
        patient.return_count += 1
        state["er_today"].append(patient)
        log_day(
            state, day,
            f"Patient {patient.patient_id} returns to ER as drop-in (critical)"
        )

Generate new daily arrivals

In [ ]:
def generate_daily_arrivals(day, state, next_patient_id):
    """
    Generate exactly 200 patients on each weekday.
    """
    for _ in range(DAILY_PATIENTS):
        patient_type = sample_patient_type()
        destination = sample_destination()
        critical = False

        if destination == "er":
            critical = sample_er_critical()

        patient = Patient(
            patient_id=next_patient_id,
            day_created=day,
            patient_type=patient_type,
            destination=destination,
            critical_status=critical,
            scheduled_day=day
        )
        next_patient_id += 1

        state["all_patients"].append(patient)
        state["patients_created"] += 1
        state["created_by_type"][patient_type] += 1
        state["created_by_destination"][destination] += 1

        # outpatient = scheduled for today
        if patient_type == "outpatient":
            add_patient_to_today_queue(patient, state)
            log_day(
                state, day,
                f"Patient {patient.patient_id} arrives as outpatient to {destination}"
            )

        # drop-in logic
        elif patient_type == "drop_in":
            if destination == "er":
                # ER patients are handled in ER queue today
                state["er_today"].append(patient)
                log_day(
                    state, day,
                    f"Patient {patient.patient_id} arrives as ER drop-in (critical={critical})"
                )
            else:
                # PCP / gyn / specialist drop-ins try today if there is room
                add_patient_to_today_queue(patient, state)
                log_day(
                    state, day,
                    f"Patient {patient.patient_id} arrives as drop-in to {destination}"
                )

    return next_patient_id

Generic provider processing

In [ ]:
def process_provider_queue(day, queue, capacity, provider_name, state):
    seen = 0
    remaining = deque()

    while queue:
        patient = queue.popleft()

        if seen < capacity:
            seen += 1
            state["seen_by_destination"][provider_name] += 1
            log_day(
                state, day,
                f"Patient {patient.patient_id} seen by {provider_name}"
            )
        else:
            patient.wait_days += 1
            state["not_seen_by_destination"][provider_name] += 1

            # If a drop-in cannot be seen, convert to scheduled outpatient
            if patient.patient_type == "drop_in":
                patient.patient_type = "outpatient"
                patient.scheduled_day = next_weekday(day)
                state["future_schedule"][patient.scheduled_day].append(patient)
                state["converted_dropin_to_outpatient"] += 1

                log_day(
                    state, day,
                    f"Patient {patient.patient_id} not seen by {provider_name}; "
                    f"converted from drop-in to outpatient on day {patient.scheduled_day}"
                )
            else:
                # outpatient who could not be seen -> reschedule
                patient.scheduled_day = next_weekday(day)
                state["future_schedule"][patient.scheduled_day].append(patient)

                log_day(
                    state, day,
                    f"Patient {patient.patient_id} outpatient not seen by {provider_name}; "
                    f"rescheduled to day {patient.scheduled_day}"
                )

ER-specific processing

In [ ]:
def process_er_queue(day, state):
    queue = state["er_today"]
    seen = 0

    while queue:
        patient = queue.popleft()

        if seen < ER_DAILY_CAPACITY:
            seen += 1
            state["seen_by_destination"]["er"] += 1
            log_day(
                state, day,
                f"Patient {patient.patient_id} seen in ER (critical={patient.critical_status})"
            )
        else:
            patient.wait_days += 1
            state["not_seen_by_destination"]["er"] += 1

            if patient.critical_status:
                return_day = next_weekday(day)
                state["er_return_next_day"][return_day].append(patient)
                state["critical_er_returned"] += 1

                log_day(
                    state, day,
                    f"Patient {patient.patient_id} critical ER not seen; returns as drop-in on day {return_day}"
                )
            else:
                patient.patient_type = "outpatient"
                patient.scheduled_day = next_weekday(day)
                state["future_schedule"][patient.scheduled_day].append(patient)
                state["noncritical_er_scheduled"] += 1

                log_day(
                    state, day,
                    f"Patient {patient.patient_id} noncritical ER not seen; scheduled outpatient on day {patient.scheduled_day}"
                )

Main daily process

In [ ]:
def daily_arrival_process(env, state):
    next_patient_id = 1

    while env.now < SIM_DAYS:
        day = int(env.now)

        # skip weekends
        if not is_weekday(day):
            yield env.timeout(1)
            continue

        log_day(state, day, f"--- DAY {day} START ---")

        # 1. release previously scheduled outpatients
        release_scheduled_patients_for_today(day, state)

        # 2. release returning critical ER patients
        release_returning_er_patients(day, state)

        # 3. generate new arrivals for today
        next_patient_id = generate_daily_arrivals(day, state, next_patient_id)

        # 4. process provider queues
        process_provider_queue(day, state["pcp_today"], PCP_DAILY_CAPACITY, "pcp", state)
        process_provider_queue(day, state["gyn_today"], GYN_DAILY_CAPACITY, "gynecologist", state)
        process_provider_queue(day, state["specialist_today"], SPECIALIST_DAILY_CAPACITY, "specialist", state)
        process_er_queue(day, state)

        # 5. snapshot queue sizes
        log_day(state, day, f"End day PCP queue size: {len(state['pcp_today'])}")
        log_day(state, day, f"End day GYN queue size: {len(state['gyn_today'])}")
        log_day(state, day, f"End day Specialist queue size: {len(state['specialist_today'])}")
        log_day(state, day, f"End day ER queue size: {len(state['er_today'])}")

        yield env.timeout(1)

Run simulation

In [ ]:
def run_simulation(sim_days=SIM_DAYS, seed=RANDOM_SEED):
    random.seed(seed)

    env = simpy.Environment()
    state = initialize_state()

    env.process(daily_arrival_process(env, state))
    env.run(until=sim_days)

    return state

Summary

In [ ]:
def print_summary(state):
    print("=" * 60)
    print("ARRIVAL / ACCESS SUMMARY")
    print("=" * 60)
    print(f"Total patients created:               {state['patients_created']}")
    print()
    print("Created by type:")
    print(f"  outpatient:                         {state['created_by_type']['outpatient']}")
    print(f"  drop_in:                            {state['created_by_type']['drop_in']}")
    print()
    print("Created by destination:")
    print(f"  pcp:                                {state['created_by_destination']['pcp']}")
    print(f"  gynecologist:                       {state['created_by_destination']['gynecologist']}")
    print(f"  specialist:                         {state['created_by_destination']['specialist']}")
    print(f"  er:                                 {state['created_by_destination']['er']}")
    print()
    print("Seen by destination:")
    print(f"  pcp:                                {state['seen_by_destination']['pcp']}")
    print(f"  gynecologist:                       {state['seen_by_destination']['gynecologist']}")
    print(f"  specialist:                         {state['seen_by_destination']['specialist']}")
    print(f"  er:                                 {state['seen_by_destination']['er']}")
    print()
    print(f"Drop-ins converted to outpatients:    {state['converted_dropin_to_outpatient']}")
    print(f"Critical ER returned next day:        {state['critical_er_returned']}")
    print(f"Noncritical ER scheduled outpatient:  {state['noncritical_er_scheduled']}")
    print()
    print(f"Outpatient showups:                   {state['outpatient_showups']}")
    print(f"Outpatient no-shows:                  {state['outpatient_noshows']}")
    print("=" * 60)

Run a short test

In [ ]:
state = run_simulation(sim_days=20, seed=42)
print_summary(state)

ARRIVAL / ACCESS SUMMARY
Total patients created:               3000

Created by type:
  outpatient:                         2082
  drop_in:                            918

Created by destination:
  pcp:                                1088
  gynecologist:                       692
  specialist:                         629
  er:                                 591

Seen by destination:
  pcp:                                600
  gynecologist:                       450
  specialist:                         300
  er:                                 375

Drop-ins converted to outpatients:    694
Critical ER returned next day:        1035
Noncritical ER scheduled outpatient:  792

Outpatient showups:                   8040
Outpatient no-shows:                  0


Inspect the log

In [ ]:
for day in range(5):
    print(f"\nDAY {day}")
    print("-" * 50)
    for msg in state["daily_log"][day][:25]:
        print(msg)


DAY 0
--------------------------------------------------
--- DAY 0 START ---
Patient 1 arrives as outpatient to pcp
Patient 2 arrives as outpatient to pcp
Patient 3 arrives as drop-in to specialist
Patient 4 arrives as drop-in to pcp
Patient 5 arrives as outpatient to pcp
Patient 6 arrives as outpatient to gynecologist
Patient 7 arrives as outpatient to pcp
Patient 8 arrives as outpatient to gynecologist
Patient 9 arrives as outpatient to gynecologist
Patient 10 arrives as drop-in to pcp
Patient 11 arrives as drop-in to specialist
Patient 12 arrives as outpatient to pcp
Patient 13 arrives as drop-in to pcp
Patient 14 arrives as outpatient to pcp
Patient 15 arrives as drop-in to specialist
Patient 16 arrives as drop-in to specialist
Patient 17 arrives as outpatient to er
Patient 18 arrives as outpatient to er
Patient 19 arrives as drop-in to gynecologist
Patient 20 arrives as drop-in to pcp
Patient 21 arrives as outpatient to pcp
Patient 22 arrives as outpatient to pcp
Patient 23 arriv